# 01 · Data Profiling

**Dataset:** Data cuaca per jam 38 ibu kota provinsi Indonesia, 2020–2024 (reanalisis ERA5 via Open-Meteo).
Detail sumber, lisensi, dan cara unduh ada di [`data/README.md`](../data/README.md).

Tujuan notebook ini adalah memahami struktur dan kualitas data mentah **sebelum** cleaning:

1. Memastikan ukuran dataset memenuhi syarat (> 1 juta baris).
2. Memeriksa skema, tipe data, dan contoh isi.
3. Memeriksa kelengkapan (missing value, jumlah jam per provinsi, kontinuitas waktu).
4. Memeriksa duplikasi dan nilai di luar rentang fisik.
5. Mendapatkan statistik deskriptif awal sebagai dasar aturan cleaning.

Polars dipakai secara **lazy** (`scan_csv`) agar data tidak dimuat penuh ke memori sebelum diperlukan.
DuckDB dipakai untuk cross-check hasil dengan SQL langsung di atas file CSV.

In [1]:
import sys
from pathlib import Path

# Notebook dapat dijalankan dari folder notebooks/ maupun dari root repository.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import duckdb
import polars as pl

from src.config import JAM_PER_PROVINSI, N_PROVINSI, RENTANG_FISIK, WEATHER_CODES

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(40)

DATA_PATH = ROOT / "data" / "raw" / "openmeteo" / "*.csv"
print("DATA_PATH :", DATA_PATH.relative_to(ROOT))
print("Polars    :", pl.__version__)
print("DuckDB    :", duckdb.__version__)

DATA_PATH : data/raw/openmeteo/*.csv
Polars    : 1.44.2
DuckDB    : 1.5.6


## 1. File dan ukuran data mentah

Data terdiri dari 38 file CSV, satu file per provinsi.

In [2]:
files = sorted(DATA_PATH.parent.glob("*.csv"))
ukuran = pl.DataFrame(
    {"file": [f.name for f in files], "ukuran_mb": [round(f.stat().st_size / 1e6, 2) for f in files]}
)
print(f"Jumlah file : {len(files)}")
print(f"Total ukuran: {ukuran['ukuran_mb'].sum():.1f} MB")
ukuran.sort("ukuran_mb", descending=True).head(5)

Jumlah file : 38
Total ukuran: 172.2 MB


file,ukuran_mb
str,f64
"""kepulauan_bangka_belitung.csv""",5.34
"""kalimantan_tengah.csv""",5.02
"""kalimantan_utara.csv""",4.89
"""kalimantan_selatan.csv""",4.85
"""kepulauan_riau.csv""",4.83


## 2. Membaca data secara lazy

`pl.scan_csv` hanya membaca header dan menyusun *query plan*; belum ada data yang dimuat.
Pola glob `*.csv` menggabungkan 38 file sekaligus.

In [3]:
lf = pl.scan_csv(str(DATA_PATH), try_parse_dates=False)
lf.collect_schema()

Schema([('provinsi', String),
        ('kota', String),
        ('latitude', Float64),
        ('longitude', Float64),
        ('timezone', String),
        ('time', String),
        ('temperature_2m', Float64),
        ('relative_humidity_2m', Int64),
        ('precipitation', Float64),
        ('weather_code', Int64),
        ('cloud_cover', Int64),
        ('surface_pressure', Float64),
        ('wind_speed_10m', Float64),
        ('wind_direction_10m', Int64)])

In [4]:
# Contoh query plan: Polars hanya akan membaca 2 kolom yang dibutuhkan (projection pushdown)
print(lf.group_by("provinsi").agg(pl.col("temperature_2m").mean()).explain())

AGGREGATE[maintain_order: false]
  [col("temperature_2m").mean()] BY [col("provinsi")]
  FROM
  Csv SCAN [/home/claude/repo/data/raw/openmeteo/aceh.csv, ... 37 other sources]
  PROJECT 2/14 COLUMNS
  ESTIMATED ROWS: 1668225


## 3. Jumlah baris (syarat tugas)

In [5]:
n_baris = lf.select(pl.len()).collect().item()
n_kolom = len(lf.collect_schema())
print(f"Jumlah baris : {n_baris:,}")
print(f"Jumlah kolom : {n_kolom}")
print(f"Syarat > 1.000.000 baris terpenuhi: {n_baris > 1_000_000}")
assert n_baris > 1_000_000

Jumlah baris : 1,666,224
Jumlah kolom : 14
Syarat > 1.000.000 baris terpenuhi: True


In [6]:
lf.head(5).collect()

provinsi,kota,latitude,longitude,timezone,time,temperature_2m,relative_humidity_2m,precipitation,weather_code,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m
str,str,f64,f64,str,str,f64,i64,f64,i64,i64,f64,f64,i64
"""Aceh""","""Banda Aceh""",5.5483,95.3238,"""Asia/Jakarta""","""2020-01-01T00:00""",26.3,86,0.0,3,94,1012.9,9.4,72
"""Aceh""","""Banda Aceh""",5.5483,95.3238,"""Asia/Jakarta""","""2020-01-01T01:00""",25.9,89,0.0,3,95,1012.3,9.8,73
"""Aceh""","""Banda Aceh""",5.5483,95.3238,"""Asia/Jakarta""","""2020-01-01T02:00""",25.6,92,0.0,3,97,1011.7,9.3,77
"""Aceh""","""Banda Aceh""",5.5483,95.3238,"""Asia/Jakarta""","""2020-01-01T03:00""",25.3,93,0.0,3,97,1010.9,8.7,83
"""Aceh""","""Banda Aceh""",5.5483,95.3238,"""Asia/Jakarta""","""2020-01-01T04:00""",25.2,93,0.0,3,98,1011.0,7.6,90


## 4. Kelengkapan per provinsi dan rentang waktu

Periode 1 Jan 2020 – 31 Des 2024 berisi 1.827 hari (termasuk 2 tahun kabisat), sehingga setiap provinsi
seharusnya punya **1.827 × 24 = 43.848 jam**.

In [7]:
lf_t = lf.with_columns(pl.col("time").str.to_datetime("%Y-%m-%dT%H:%M"))

per_prov = (
    lf_t.group_by("provinsi", "kota", "timezone")
    .agg(
        pl.len().alias("n_jam"),
        pl.col("time").min().alias("mulai"),
        pl.col("time").max().alias("selesai"),
        pl.col("latitude").first(),
        pl.col("longitude").first(),
    )
    .with_columns((pl.col("n_jam") == JAM_PER_PROVINSI).alias("lengkap"))
    .sort("provinsi")
    .collect()
)
print(f"Provinsi       : {per_prov.height} (target {N_PROVINSI})")
print(f"Semua lengkap  : {per_prov['lengkap'].all()}")
per_prov

Provinsi       : 38 (target 38)
Semua lengkap  : True


provinsi,kota,timezone,n_jam,mulai,selesai,latitude,longitude,lengkap
str,str,str,u32,datetime[μs],datetime[μs],f64,f64,bool
"""Aceh""","""Banda Aceh""","""Asia/Jakarta""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,5.5483,95.3238,true
"""Bali""","""Denpasar""","""Asia/Makassar""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,-8.6705,115.2126,true
"""Banten""","""Serang""","""Asia/Jakarta""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,-6.1104,106.164,true
"""Bengkulu""","""Bengkulu""","""Asia/Jakarta""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,-3.8004,102.2655,true
"""DI Yogyakarta""","""Yogyakarta""","""Asia/Jakarta""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,-7.7956,110.3695,true
"""DKI Jakarta""","""Jakarta""","""Asia/Jakarta""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,-6.2088,106.8456,true
"""Gorontalo""","""Gorontalo""","""Asia/Makassar""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,0.5435,123.0568,true
"""Jambi""","""Jambi""","""Asia/Jakarta""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,-1.6101,103.6131,true
"""Jawa Barat""","""Bandung""","""Asia/Jakarta""",43848,2020-01-01 00:00:00,2024-12-31 23:00:00,-6.9175,107.6191,true


In [8]:
# Kontinuitas waktu: selisih antar-baris berurutan di setiap provinsi harus tepat 1 jam
celah = (
    lf_t.sort("provinsi", "time")
    .with_columns(pl.col("time").diff().over("provinsi").alias("selisih"))
    .filter(pl.col("selisih").is_not_null() & (pl.col("selisih") != pl.duration(hours=1)))
    .select(pl.len())
    .collect()
    .item()
)
print(f"Jumlah celah/lompatan waktu: {celah}")

Jumlah celah/lompatan waktu: 0


## 5. Missing value dan duplikasi

In [9]:
nulls = lf.null_count().collect().transpose(include_header=True, header_name="kolom", column_names=["n_null"])
nulls.with_columns((pl.col("n_null") / n_baris * 100).round(4).alias("persen"))

kolom,n_null,persen
str,u32,f64
"""provinsi""",0,0.0
"""kota""",0,0.0
"""latitude""",0,0.0
"""longitude""",0,0.0
"""timezone""",0,0.0
"""time""",0,0.0
"""temperature_2m""",0,0.0
"""relative_humidity_2m""",0,0.0
"""precipitation""",0,0.0


In [10]:
dup_baris = n_baris - lf.unique().select(pl.len()).collect().item()
dup_kunci = n_baris - lf.unique(subset=["provinsi", "time"]).select(pl.len()).collect().item()
print(f"Duplikat baris identik          : {dup_baris}")
print(f"Duplikat kunci (provinsi, time) : {dup_kunci}")

Duplikat baris identik          : 0
Duplikat kunci (provinsi, time) : 0


## 6. Statistik deskriptif

In [11]:
num_cols = [
    "temperature_2m", "relative_humidity_2m", "precipitation", "cloud_cover",
    "surface_pressure", "wind_speed_10m", "wind_direction_10m",
]
lf.select(num_cols).collect().describe(percentiles=(0.01, 0.25, 0.5, 0.75, 0.99))

statistic,temperature_2m,relative_humidity_2m,precipitation,cloud_cover,surface_pressure,wind_speed_10m,wind_direction_10m
str,f64,f64,f64,f64,f64,f64,f64
"""count""",1.666224e6,1.666224e6,1.666224e6,1.666224e6,1.666224e6,1.666224e6,1.666224e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",27.19243,84.280603,0.349394,81.546081,1001.363779,6.694755,179.410648
"""std""",2.611812,10.413966,0.953423,26.570003,29.174698,4.866627,98.338359
"""min""",12.5,20.0,0.0,0.0,833.4,0.0,0.0
"""1%""",18.8,52.0,0.0,3.0,839.5,0.7,6.0
"""25%""",25.6,78.0,0.0,72.0,1005.9,3.3,108.0
"""50%""",27.0,87.0,0.0,96.0,1008.2,5.4,169.0
"""75%""",28.8,92.0,0.2,100.0,1009.9,8.6,263.0


## 7. Pemeriksaan rentang fisik

Batas rentang didefinisikan di `src/config.py` (`RENTANG_FISIK`). Nilai di luar batas ini dianggap
**tidak valid** (error), berbeda dengan outlier yang mungkin saja peristiwa nyata.

In [12]:
cek_rentang = lf.select(
    [
        (~pl.col(c).is_between(lo, hi)).sum().alias(c)
        for c, (lo, hi) in RENTANG_FISIK.items()
    ]
).collect()
cek_rentang.transpose(include_header=True, header_name="kolom", column_names=["n_di_luar_rentang"])

kolom,n_di_luar_rentang
str,u32
"""temperature_2m""",0
"""relative_humidity_2m""",0
"""precipitation""",0
"""cloud_cover""",0
"""surface_pressure""",0
"""wind_speed_10m""",0
"""wind_direction_10m""",0


In [13]:
# Kode cuaca WMO: semua kode harus dikenal
kode = (
    lf.group_by("weather_code").len().sort("weather_code").collect()
    .with_columns(
        pl.col("weather_code").replace_strict(WEATHER_CODES, default="TIDAK DIKENAL").alias("keterangan"),
        (pl.col("len") / n_baris * 100).round(2).alias("persen"),
    )
)
kode

weather_code,len,keterangan,persen
i64,u32,str,f64
0,76391,"""Cerah""",4.58
1,131307,"""Umumnya cerah""",7.88
2,145516,"""Berawan sebagian""",8.73
3,586621,"""Berawan/mendung""",35.21
51,433594,"""Gerimis ringan""",26.02
53,124092,"""Gerimis sedang""",7.45
55,38646,"""Gerimis lebat""",2.32
61,74424,"""Hujan ringan""",4.47
63,51149,"""Hujan sedang""",3.07


In [14]:
# Konsistensi antar-kolom: kode gerimis/hujan (>= 51) seharusnya disertai presipitasi > 0
inkonsisten = lf.select(
    ((pl.col("weather_code") >= 51) & (pl.col("precipitation") == 0)).sum().alias("kode_hujan_tanpa_presipitasi"),
    ((pl.col("weather_code") < 51) & (pl.col("precipitation") > 0)).sum().alias("presipitasi_tanpa_kode_hujan"),
).collect()
inkonsisten

kode_hujan_tanpa_presipitasi,presipitasi_tanpa_kode_hujan
u32,u32
0,0


## 8. Pemeriksaan nilai yang tampak ekstrem

Tekanan permukaan minimum 833 hPa dan suhu minimum 12,5 °C terlihat janggal untuk wilayah tropis.
Kita periksa apakah ini error atau memang karakteristik lokasi.

In [15]:
(
    lf.group_by("provinsi", "kota")
    .agg(
        pl.col("surface_pressure").mean().round(1).alias("tekanan_rata2"),
        pl.col("temperature_2m").mean().round(2).alias("suhu_rata2"),
        pl.col("temperature_2m").min().alias("suhu_min"),
        pl.col("precipitation").max().alias("hujan_maks_per_jam"),
    )
    .sort("tekanan_rata2")
    .head(5)
    .collect()
)

provinsi,kota,tekanan_rata2,suhu_rata2,suhu_min,hujan_maks_per_jam
str,str,f64,f64,f64,f64
"""Papua Pegunungan""","""Wamena""",840.0,20.33,12.5,9.8
"""Jawa Barat""","""Bandung""",934.8,24.06,17.2,18.0
"""DI Yogyakarta""","""Yogyakarta""",998.8,26.26,17.4,22.8
"""Lampung""","""Bandar Lampung""",999.0,26.85,22.3,15.5
"""Nusa Tenggara Timur""","""Kupang""",999.0,27.84,19.8,24.4


## 9. Cross-check dengan DuckDB

DuckDB bisa langsung membaca file CSV dengan SQL tanpa memuatnya ke pandas.
Hasilnya harus sama dengan perhitungan Polars di atas.

In [16]:
con = duckdb.connect()
con.sql(f"""
    SELECT
        COUNT(*)                    AS n_baris,
        COUNT(DISTINCT provinsi)    AS n_provinsi,
        MIN(time)                   AS mulai,
        MAX(time)                   AS selesai,
        ROUND(SUM(precipitation) / COUNT(DISTINCT provinsi) / 5, 1) AS rata2_hujan_tahunan_mm
    FROM read_csv_auto('{DATA_PATH.as_posix()}')
""").pl()

n_baris,n_provinsi,mulai,selesai,rata2_hujan_tahunan_mm
i64,i64,datetime[μs],datetime[μs],f64
1666224,38,2020-01-01 00:00:00,2024-12-31 23:00:00,3064.0


## 10. Ringkasan temuan profiling

| Aspek | Hasil |
|---|---|
| Ukuran | **1.666.224 baris**, 14 kolom, 38 file CSV (172 MB), memenuhi syarat > 1 juta baris |
| Kelengkapan | 38 provinsi × 43.848 jam, tidak ada celah waktu |
| Missing value | 0 di semua kolom |
| Duplikasi | 0 baris identik, 0 duplikat kunci `(provinsi, time)` |
| Rentang fisik | Tidak ada nilai di luar rentang; kode cuaca semuanya valid (10 kode, tanpa badai petir karena ERA5 tidak memodelkan petir) |
| Konsistensi | Kode cuaca hujan/gerimis selalu disertai presipitasi > 0, dan sebaliknya |
| Nilai ekstrem | Tekanan ±840 hPa dan suhu minimum 12,5 °C berasal dari **Wamena (Papua Pegunungan)** yang berada di dataran tinggi ±1.550 mdpl, jadi bukan error |
| Distribusi hujan | Sangat miring ke kanan: median 0 mm, persentil 99 sekitar 4,7 mm/jam, maksimum 31,3 mm/jam |

**Implikasi untuk cleaning (notebook 02):**

- Tipe data perlu distandarkan (`time` masih string, numerik bisa diperkecil ke `Float32`/`Int16` untuk hemat memori).
- Walaupun data sudah bersih, pipeline tetap disusun **defensif** (deduplikasi, validasi rentang, imputasi) agar aman bila data diunduh ulang.
- Outlier perlu dideteksi **per provinsi**, bukan global, karena perbedaan elevasi membuat Wamena wajar lebih dingin.
- Hujan ekstrem diberi flag, tidak dihapus, karena merupakan peristiwa nyata yang penting untuk analisis.
- Data perlu dikonversi ke **Parquet** agar analisis selanjutnya lebih cepat.